In [1]:
# === assemble_results (robust paths) ===
import pandas as pd
from pathlib import Path

# auto-root: if running inside notebooks/, go up one level
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

TAB = ROOT / "reports" / "tables"
TAB.mkdir(parents=True, exist_ok=True)

def must(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return path

print("Using tables dir:", TAB)

# 1) Mains
metrics_path = must(TAB / "metrics.csv")
metrics = pd.read_csv(metrics_path)
keep = metrics[metrics["strategy"].isin(["risk_parity","ts_momentum"])].copy()
main = keep[["strategy","CAGR","Volatility","Sharpe","MaxDrawdown"]].round(4)
main_out = TAB / "results_main.csv"
main.to_csv(main_out, index=False)
print("Saved:", main_out)

# 2) 0/10/20 bps
costs_path = must(TAB / "sensitivity_costs.csv")
costs = pd.read_csv(costs_path)
cost_tbl = costs.pivot(index="cost_bps", columns="strategy", values="Sharpe").round(3)
cost_out = TAB / "robustness_costs_sharpe.csv"
cost_tbl.to_csv(cost_out)
print("Saved:", cost_out)

# 3) Walk-Forward
wf_path = must(TAB / "walkforward_metrics.csv")
wf = pd.read_csv(wf_path)
test = wf[wf["split"]=="test"].copy()
agg_mean = test.groupby("strategy")[["CAGR","Volatility","Sharpe","MaxDrawdown"]].mean()
agg_std  = test.groupby("strategy")[["CAGR","Volatility","Sharpe","MaxDrawdown"]].std()
summary = (agg_mean.round(4)).astype(str) + " ± " + (agg_std.round(4)).astype(str)
wf_out = TAB / "walkforward_test_summary.csv"
summary.to_csv(wf_out)
print("Saved:", wf_out)

Using tables dir: /Users/jasper/Downloads/My Dissertation/robo-advisor-offline/reports/tables
Saved: /Users/jasper/Downloads/My Dissertation/robo-advisor-offline/reports/tables/results_main.csv
Saved: /Users/jasper/Downloads/My Dissertation/robo-advisor-offline/reports/tables/robustness_costs_sharpe.csv
Saved: /Users/jasper/Downloads/My Dissertation/robo-advisor-offline/reports/tables/walkforward_test_summary.csv
